<a href="https://colab.research.google.com/github/rauf-mifteev/Ahuntsic_AI_Optimization/blob/main/AI_TP_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intelligence Artificielle 1 — Travail Pratique 2

Optimisation : Programmation Linéaire, Glouton et Programmation Dynamique

## Partie 1 — Programmation linéaire avec PuLP

Cette première section utilise la bibliothèque PuLP pour formuler le problème mathématiquement et garantir la sélection optimale exacte des équipes.

### 1.1 Données du problème

In [1]:
joueurs = [
    {"nom": "Alice", "score": 88, "salaire": 1200, "poids": 72},
    {"nom": "Bob", "score": 91, "salaire": 1800, "poids": 85},
    {"nom": "Clara", "score": 84, "salaire": 950, "poids": 68},
    {"nom": "David", "score": 93, "salaire": 2100, "poids": 90},
    {"nom": "Emma", "score": 79, "salaire": 300, "poids": 65},
    {"nom": "Frank", "score": 87, "salaire": 2400, "poids": 95},
    {"nom": "Grace", "score": 85, "salaire": 1050, "poids": 70},
    {"nom": "Hugo", "score": 89, "salaire": 1600, "poids": 80}
]

### 1.2 Définition du problème de maximisation

In [2]:
import pulp

prob = pulp.LpProblem("Ligue_Basketball", pulp.LpMaximize)

### 1.3 Variables de décision binaires (0 ou 1)

In [3]:
# x_A[nom] = 1 si le joueur est dans l'équipe A
x_A = pulp.LpVariable.dicts("EqA", [j["nom"] for j in joueurs], cat='Binary')
x_B = pulp.LpVariable.dicts("EqB", [j["nom"] for j in joueurs], cat='Binary')

### 1.4 Fonction objective : Maximiser le score total

In [4]:
prob += pulp.lpSum([j["score"] * (x_A[j["nom"]] + x_B[j["nom"]]) for j in joueurs]), "Score_Total"

### 1.5 Contraintes

In [5]:
# Exactement 3 joueurs par équipe
prob += pulp.lpSum([x_A[j["nom"]] for j in joueurs]) == 3, "Taille_EqA"
prob += pulp.lpSum([x_B[j["nom"]] for j in joueurs]) == 3, "Taille_EqB"

# Un joueur ne peut être que dans une seule équipe (ou aucune)
for j in joueurs:
    prob += x_A[j["nom"]] + x_B[j["nom"]] <= 1, f"Unicite_{j['nom']}"

# Budget total max de 8500$
prob += pulp.lpSum([j["salaire"] * (x_A[j["nom"]] + x_B[j["nom"]]) for j in joueurs]) <= 8500, "Budget_Total"

# Poids max de 250 Kg par équipe
prob += pulp.lpSum([j["poids"] * x_A[j["nom"]] for j in joueurs]) <= 250, "Poids_EqA"
prob += pulp.lpSum([j["poids"] * x_B[j["nom"]] for j in joueurs]) <= 250, "Poids_EqB"

### 1.6 Résolution

In [6]:
prob.solve()

1

### 1.7 Affichage des résultats

In [7]:
print(f"Statut : {pulp.LpStatus[prob.status]}\n")

equipe_A, equipe_B = [], []
score_A, score_B = 0, 0

for j in joueurs:
    if x_A[j["nom"]].varValue == 1:
        equipe_A.append(j["nom"])
        score_A += j["score"]
    elif x_B[j["nom"]].varValue == 1:
        equipe_B.append(j["nom"])
        score_B += j["score"]

print(f"Équipe A : {equipe_A} (Score: {score_A})")
print(f"Équipe B : {equipe_B} (Score: {score_B})")
print(f"Score Total Optimal : {pulp.value(prob.objective)} points")

Statut : Optimal

Équipe A : ['Emma', 'Grace', 'Hugo'] (Score: 253)
Équipe B : ['Alice', 'Bob', 'David'] (Score: 272)
Score Total Optimal : 525.0 points


## Partie 2 — Algorithme glouton
Cette section explore trois stratégies gloutonnes pour former les équipes rapidement, afin de comparer l'efficacité de ces choix locaux avec la solution mathématique optimale. En gardant les même données 'joueurs' que dans la partie 1

In [8]:
def verifier_contraintes(equipe, nouveau_joueur, budget_restant):
    """Fonction modulaire pour vérifier si un joueur peut être ajouté."""
    poids_actuel = sum(j["poids"] for j in equipe)
    if len(equipe) >= 3:
        return False
    if poids_actuel + nouveau_joueur["poids"] > 250:
        return False
    if nouveau_joueur["salaire"] > budget_restant:
        return False
    return True

def algorithme_glouton(joueurs_dispo, strategie_tri):
    """Exécute l'algorithme glouton basé sur une clé de tri spécifique."""
    # Trier les joueurs selon la stratégie
    joueurs_tries = sorted(joueurs_dispo, key=strategie_tri, reverse=True)

    equipe_A, equipe_B = [], []
    budget_restant = 8500

    for joueur in joueurs_tries:
        # Tenter d'ajouter à l'équipe A d'abord
        if verifier_contraintes(equipe_A, joueur, budget_restant):
            equipe_A.append(joueur)
            budget_restant -= joueur["salaire"]
        # Sinon, tenter d'ajouter à l'équipe B
        elif verifier_contraintes(equipe_B, joueur, budget_restant):
            equipe_B.append(joueur)
            budget_restant -= joueur["salaire"]

    # Vérification finale
    if len(equipe_A) < 3 or len(equipe_B) < 3:
        print("Erreur : Impossible de remplir les équipes avec ces contraintes.")

    score_total = sum(j["score"] for j in equipe_A + equipe_B)
    budget_utilise = 8500 - budget_restant
    return equipe_A, equipe_B, score_total, budget_utilise

### Stratégie 1: Meilleur score absolu

In [9]:
print("--- Stratégie 1 : Meilleur score ---")
eqA1, eqB1, score1, budget1 = algorithme_glouton(joueurs, lambda j: j["score"])
print(f"Équipes: {[j['nom'] for j in eqA1]} | {[j['nom'] for j in eqB1]}")
print(f"Score: {score1}, Budget: {budget1}$\n")

--- Stratégie 1 : Meilleur score ---
Équipes: ['David', 'Bob', 'Alice'] | ['Hugo', 'Grace', 'Emma']
Score: 525, Budget: 8050$



### Stratégie 2: Meilleur ratio score/salaire

In [10]:
print("--- Stratégie 2 : Meilleur ratio score/salaire ---")
eqA2, eqB2, score2, budget2 = algorithme_glouton(joueurs, lambda j: j["score"] / j["salaire"])
print(f"Équipes: {[j['nom'] for j in eqA2]} | {[j['nom'] for j in eqB2]}")
print(f"Score: {score2}, Budget: {budget2}$\n")

--- Stratégie 2 : Meilleur ratio score/salaire ---
Équipes: ['Emma', 'Clara', 'Grace'] | ['Alice', 'Hugo', 'Bob']
Score: 516, Budget: 6900$



### Stratégie 3: Meilleur ratio score/poids

In [11]:
print("--- Stratégie 3 : Meilleur ratio score/poids ---")
eqA3, eqB3, score3, budget3 = algorithme_glouton(joueurs, lambda j: j["score"] / j["poids"])
print(f"Équipes: {[j['nom'] for j in eqA3]} | {[j['nom'] for j in eqB3]}")
print(f"Score: {score3}, Budget: {budget3}$\n")

--- Stratégie 3 : Meilleur ratio score/poids ---
Équipes: ['Clara', 'Alice', 'Emma'] | ['Grace', 'Hugo', 'Bob']
Score: 516, Budget: 6900$



## Partie 3 — Récursivité et programmation dynamique

Cette section met de côté les contraintes de la ligue pour se concentrer uniquement sur les concepts de récursivité et de programmation dynamique.

### 3.1  Score cumulé récursif

In [16]:
# Trier la liste une seule fois à l'extérieur de la fonction
joueurs_tries = sorted(joueurs, key=lambda j: j["score"], reverse=True)

def score_cumule(joueurs_liste, k):
    """Calcule le score total des k meilleurs joueurs de façon récursive."""
    # Cas d'arrêt (Base case)
    if k == 0:
        print("score_cumule(joueurs, 0) = 0")
        return 0

    # Appel récursif (Recursive step)
    score_precedent = score_cumule(joueurs_liste, k-1)

    # Récupération du joueur actuel (k-1 car les index commencent à 0)
    joueur_actuel = joueurs_liste[k-1]
    score_total = joueur_actuel["score"] + score_precedent

    # Affichage de l'étape actuelle
    print(f"score_cumule(joueurs, {k}) = {score_precedent} + {joueur_actuel['score']} ({joueur_actuel['nom']}) = {score_total}")

    return score_total

# --- Test de la fonction ---
print("--- Test Score Cumulé ---")
score_cumule(joueurs_tries, 2)

--- Test Score Cumulé ---
score_cumule(joueurs, 0) = 0
score_cumule(joueurs, 1) = 0 + 93 (David) = 93
score_cumule(joueurs, 2) = 93 + 91 (Bob) = 184


184

### 3.2 Fibonacci des scores (Naïf vs Mémoïsé)

In [13]:
import time

# Les valeurs de départ proviennent des deux meilleurs joueurs de PuLP
FIB_0 = 93 # David
FIB_1 = 91 # Bob

def fib_naif(n):
    """Calcule Fibonacci sans mémorisation (très lent pour les grands n)."""
    if n == 0:
        return FIB_0
    if n == 1:
        return FIB_1
    return fib_naif(n-1) + fib_naif(n-2)

# Dictionnaire global pour stocker les résultats
memo = {}

def fib_memo(n):
    """Calcule Fibonacci avec mémorisation (Programmation Dynamique)."""
    if n in memo:
        return memo[n] # Retourne le résultat sauvegardé

    if n == 0:
        resultat = FIB_0
    elif n == 1:
        resultat = FIB_1
    else:
        resultat = fib_memo(n-1) + fib_memo(n-2)

    memo[n] = resultat # Sauvegarde avant de retourner
    return resultat

# --- Mesure des temps d'exécution ---
n_test = 35
print(f"--- Comparaison Fibonacci pour n={n_test} ---")

# Test Naïf
debut_naif = time.perf_counter()
res_naif = fib_naif(n_test)
fin_naif = time.perf_counter()
print(f"fib_naif({n_test}) = {res_naif}")
print(f"Temps naïf: {fin_naif - debut_naif:.6f}s")

# Test Mémoïsé
debut_memo = time.perf_counter()
res_memo = fib_memo(n_test)
fin_memo = time.perf_counter()
print(f"fib_memo({n_test}) = {res_memo}")
print(f"Temps mémoïsé: {fin_memo - debut_memo:.6f}s")

--- Comparaison Fibonacci pour n=35 ---
fib_naif(35) = 1370067806
Temps naïf: 5.183881s
fib_memo(35) = 1370067806
Temps mémoïsé: 0.000164s
